# Marine Environmental Monitoring Measurements from Estuarine and Coastal Zones of the Basque Country (1995–2014) Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR^2 marine environmental monitoring dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.07je-gjna/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant JSON-LD schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.07je-gjna/fair2.json"

# Load dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata object (do NOT subscript or iterate)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}\n")
print(f"Identifier: {metadata.identifier}\n")
print(f"Temporal coverage: {metadata.temporalCoverage}\n")
print(f"Spatial coverage: {metadata.spatialCoverage}\n")

## 2. Data Overview
List all available record sets, fields, and their `@id` values, which uniquely identify entities in the dataset.
We use the dataset's Croissant schema structure to enumerate record sets and fields by `@id`.

In [ ]:
# List all available record sets in the dataset
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'unknown')}")
    # List fields for each record set
    if 'fields' in rs:
        print("  Fields:")
        for field in rs['fields']:
            print(f"    - @id: {field['@id']}, name: {field.get('name', 'unknown')}, dataType: {field.get('dataType', '')}")
    print("---")

## 3. Data Extraction
Load records for selected record sets into Pandas DataFrames.
All record sets and fields are referenced by their `@id`, ensuring reproducibility and clarity.

We demonstrate data extraction for the first two record sets, as examples.

In [ ]:
# Prepare list of record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids[:2]:  # Example: Only extract from first two for demonstration
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Columns for record set {rs_id}:\n", dataframes[rs_id].columns.tolist())
        display(dataframes[rs_id].head())
    else:
        print(f"No records found for record set {rs_id}")

## 4. Exploratory Data Analysis (EDA)
Demonstrate basic filtering, normalization, and grouping for a selected numeric field within one of the record sets.

**Note:** For demonstration, we will inspect field data types and select a numeric column for EDA if available.

In [ ]:
import numpy as np

# Select the first record set with valid records
for rs_id, df in dataframes.items():
    if not df.empty:
        primary_df = df
        primary_rs_id = rs_id
        break

# Find numeric fields from metadata
numeric_fields = []
for rs in dataset.record_sets:
    if rs['@id'] == primary_rs_id:
        for field in rs['fields']:
            if field.get('dataType', '').lower() in ['integer', 'float', 'number']:
                numeric_fields.append(field['@id'])
        break

# Use first numeric field and a group/categorical field if present
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field for EDA: {numeric_field_id}")
else:
    print("No numeric fields found.")
    numeric_field_id = None

# Try to find a categorical field to group by
group_field_id = None
for col in primary_df.columns:
    if col != numeric_field_id and primary_df[col].dtype == 'object':
        group_field_id = col
        print(f"Grouping by field: {group_field_id}")
        break

if numeric_field_id and numeric_field_id in primary_df.columns:
    # Filter for values greater than a threshold (e.g., 10)
    threshold = 10
    filtered_df = primary_df[primary_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field_id and compute mean if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("Numeric field for EDA not available in DataFrame.")

## 5. Visualization
Visualize numeric distributions and relationships using matplotlib.

Below, we create a histogram and a boxplot for the chosen numeric field, and a bar chart for grouping.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram and boxplot
if numeric_field_id and numeric_field_id in filtered_df.columns:
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)

    plt.subplot(1, 2, 2)
    sns.boxplot(filtered_df[numeric_field_id].dropna())
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.xlabel(numeric_field_id)

    plt.tight_layout()
    plt.show()

    # Bar chart of grouped means
    if group_field_id and group_field_id in grouped_df.columns:
        plt.figure(figsize=(10, 4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
Using the FAIR^2 marine monitoring dataset and the `mlcroissant` library, we've:
- loaded metadata and records from the Croissant schema,
- explored available record sets and fields (referenced by `@id`),
- extracted and inspected data as Pandas DataFrames,
- demonstrated filtering, normalization, and grouping operations,
- visualized numeric distributions and grouped means.

This notebook provides a framework for further marine environmental analysis. Be sure to adjust the field and record set selections to explore the full scope of the dataset.